# 13 - Robustez del ajuste de hiperparámetros
En el notebook 12 hemos visto que en los modelos de árboles (`random_forest`, `lightgbm` y `xgboost_balanced`), el ajuste de hiperparámetros no ha mejorado a los valores por defecto. Vamos a intentar comprobar por qué. Puede ser por dos motivos:
1. La búsqueda se queda corta, porque solo habíamos probado 30 candidatas al azar.
2. La validación interna es muy ruidosa, porque la clase A tiene unos 24 tramos por ciudad. La candidata que gana puede ser la que ha tenido más suerte, y entonces una búsqueda mejor no ayudaría.

Vamos a repetir el ajuste de hiperparámetros de los árboles de dos maneras y lo comparamos con el ajuste principal (al que llamamos A).
- **B**, con otra seed, que cambia los pliegues internos y las candidatas. Si el resultado cambia mucho, apunta a la segunda explicación.
- **C**, con 100 candidatas, en lugar de 30. Si el test no mejora, la primera explicación no se sostiene.

No vamos a repetir la regresión logística, porque su rejilla siempre prueba las mismas combinaciones, y los resultados han sido mejores.

Los ajustes se lanzan con `make tune-variants` (`urban_accessibility_ml/modeling/tuning/tuning_variants.py`), así que aquí solo se leen los resultados.

## Carga de los resultados

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from urban_accessibility_ml.config import REPORTS_DIR, SEED
from urban_accessibility_ml.modeling.tuning.tuning_constants import TUNING_N_CANDIDATES, TUNING_TIE_TOLERANCE
from urban_accessibility_ml.modeling.tuning.tuning_variants import TREE_MODELS, VARIANTS, VARIANTS_DIR
from urban_accessibility_ml.plotting.palettes import SERIE_BASE
from urban_accessibility_ml.plotting.plots import SERIES, save_figure, use_style

RESULTS = REPORTS_DIR / "results"
KEY = ["strategy", "name", "fold", "feature_set", "profile", "model"]
# One colour per model, the same as in notebook 11
MODEL_COLORS = {"random_forest": SERIES[0], "xgboost_balanced": SERIES[1], "lightgbm": SERIES[2]}

use_style()

default = pd.read_csv(RESULTS / "experiment-matrix-default.csv")
matrices = {
    "A (30 candidatas)": pd.read_csv(RESULTS / "experiment-matrix-tuned.csv"),
    "B (otra seed)": pd.read_csv(VARIANTS_DIR / "experiment-matrix_seed2.csv"),
    "C (100 candidatas)": pd.read_csv(VARIANTS_DIR / "experiment-matrix_n100.csv"),
}
best_params = {
    "A": json.load(open(RESULTS / "best_params.json")),
    "B": json.load(open(VARIANTS_DIR / "best_params_seed2.json")),
    "C": json.load(open(VARIANTS_DIR / "best_params_n100.json")),
}
candidates = {
    "A": pd.read_csv(RESULTS / "tuning-candidates.csv"),
    "B": pd.read_csv(VARIANTS_DIR / "tuning-candidates_seed2.csv"),
    "C": pd.read_csv(VARIANTS_DIR / "tuning-candidates_n100.csv"),
}
candidates = {name: table[~table["feature_set"].str.contains("@")] for name, table in candidates.items()}


def f1_by_experiment(results):
    """F1 macro of each experiment of the tree models, indexed by KEY."""
    return results[results["model"].isin(TREE_MODELS)].set_index(KEY)["f1_macro"]


f1_default = f1_by_experiment(default)
f1_tuned = {name: f1_by_experiment(r).reindex(f1_default.index) for name, r in matrices.items()}
print(f"{len(f1_default)} experimentos de árboles en cada matriz")

## Las variantes
Las tres variantes solo se diferencian en la seed y en el número de candidatas.

In [ ]:
pd.DataFrame(
    {
        "seed": [SEED, VARIANTS["seed2"]["seed"], SEED],
        "candidatas": [TUNING_N_CANDIDATES, VARIANTS["seed2"]["n_candidates"], VARIANTS["n100"]["n_candidates"]],
    },
    index=["A (30 candidatas)", "B (otra seed)", "C (100 candidatas)"],
)

## Resultados en el test
Se han calculado la métrica F1 macro en el test de cada árbol, con los valores por defecto y con cada ajuste (media de todos los experimentos).

La segunda tabla es la diferencia con los valores por defecto (sin aplicar hiperparámetros). Es positiva si el ajuste mejora.

In [ ]:
mean_f1 = pd.DataFrame({"por defecto": f1_default.groupby("model").mean()} | {n: s.groupby("model").mean() for n, s in f1_tuned.items()})
mean_f1.round(4)

In [ ]:
difference = mean_f1.drop(columns="por defecto").sub(mean_f1["por defecto"], axis=0)
difference.round(4)

Mostramos los resultados en un gráfico.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
width = 0.26
x = np.arange(len(difference))

for i, (name, color) in enumerate(zip(difference.columns, SERIES)):
    ax.bar(x + (i - 1) * width, difference[name], width, color=color, label=name)
ax.axhline(0, color=SERIE_BASE, linewidth=1, linestyle="--")
ax.set_xticks(x, difference.index)
ax.set_ylabel("F1 macro ajustado menos por defecto (test)")
ax.legend()
plt.tight_layout()
save_figure("diferencia_por_variante", "0.13-robustez-ajuste", fig=fig)
plt.show()

Ninguna de las tres ejecuciones supera a los valores por defecto. Con 100 candidatas (C) no mejora, así que la búsqueda no parece quedarse corta. Y con otra seed (B) el resultado ya cambia respecto a A, lo que apunta al ruido.

## Cambio del resultado según la seed
Vamos a comparar, experimento a experimento, el ajuste A con los valores por defecto y con el ajuste B, que solo cambia la seed.

In [ ]:
a = f1_tuned["A (30 candidatas)"]
gap = pd.DataFrame(
    {
        "A frente a por defecto": (a - f1_default).abs().groupby("model").mean(),
        "A frente a B (solo cambia la seed)": (a - f1_tuned["B (otra seed)"]).abs().groupby("model").mean(),
    }
)
gap.round(4)

Entre A y B, que solo cambian la seed, la diferencia es de unos 0,008 a 0,010 de F1 macro, parecida a la que hay entre A y los valores por defecto (de 0,007 a 0,016).
Cambiar la seed mueve los resultados tanto como el propio ajuste.

## Variabilidad de los valores elegidos
Comprobamos cuántos de los 459 ajustes de los árboles tienen los mismos hiperparámetros en dos variantes.

In [ ]:
tree_keys = list(best_params["B"])
same = pd.Series(
    {
        "A y B (otra seed)": sum(best_params["A"][k] == best_params["B"][k] for k in tree_keys),
        "A y C (100 candidatas)": sum(best_params["A"][k] == best_params["C"][k] for k in tree_keys),
    },
    name=f"coinciden de {len(tree_keys)}",
)
same.to_frame()

Al cambiar de seed (B) no coincide ningún valor de los hiperparámetros, y al subir a 100 candidatas (C) solo coinciden 52.

Esto no es una prueba de ruido. Con otra seed se generan candidatas distintas y los espacios de búsqueda son continuos y aleatorios, así que no coincidir es normal.

## Comparación de la F1 interna con la del test
Al ajustar, puntuamos cada candidata con una validación dentro de los datos de entrenamiento (la F1 macro interna) y nos quedamos con la que mejor sale. Después medimos la elegida en el test, que no ha visto nunca (la F1 macro del test).

Si la validación interna fuera fiable, una candidata con mejor F1 interna debería dar también mejor F1 en el test. Si no pasa, es que la validación es ruidosa y estamos eligiendo por suerte.

La tabla muestra las dos F1 de la candidata elegida, como media de cada modelo.

In [ ]:
def internal_score(table):
    """Mean inner-validation F1 macro of the chosen candidate, per tree model."""
    chosen = table[table["selected"] & table["model"].isin(TREE_MODELS)]
    return chosen.groupby("model")["f1_macro_mean"].mean()


inner = pd.DataFrame({name: internal_score(candidates[name[0]]) for name in matrices})
test = pd.DataFrame({name: s.groupby("model").mean() for name, s in f1_tuned.items()})
pd.concat({"F1 macro interna (validación)": inner, "F1 macro en el test": test}, axis=1).round(4)

Entre A y B la F1 interna casi no cambia (en LightGBM es 0,897 en las dos), pero la del test sí, así que la interna no nos sirve para saber qué pasará en el test.

Con 100 candidatas (C) la F1 interna sube (0,904) y la del test empeora, es decir, que buscando más solo hemos encontrado candidatas que puntúan mejor en la validación interna.

Por lo que parece que la búsqueda se ajusta al ruido de la validación interna.

## Validación interna de los cuatro modelos
Vamos a comparar los cuatro modelos con la validación interna del ajuste A, que solo usa datos de entrenamiento (`tuning-candidates.csv`), sin tocar el test.

Para cada una de las 17 particiones del conjunto `completo`, usamos la F1 macro interna de la candidata elegida de cada modelo. Luego revisamos en cuántas particiones es la más alta, teniendo en cuenta los empates.

In [ ]:
chosen = candidates["A"][(candidates["A"]["feature_set"] == "completo") & candidates["A"]["selected"]]
inner_f1 = chosen.pivot(index=["strategy", "name", "fold"], columns="model", values="f1_macro_mean")
wins = inner_f1.eq(inner_f1.max(axis=1), axis=0).sum()
pd.DataFrame(
    {
        "F1 macro interna": inner_f1.mean(),
        "desviación típica interna": chosen.groupby("model")["f1_macro_std"].mean(),
        f"particiones en las que es la más alta (de {len(inner_f1)})": wins,
    }
).round(3)

Como podemos ver, ningún modelo destaca claramente. La regresión logística tiene la F1 macro interna más alta en 9 de las 17 particiones, LightGBM en 6 y `random_forest` en 3. XGBoost no la tiene más alta en ninguna. Los empates cuentan para todos los modelos empatados, por eso la suma es 18.

Las medias son muy parecidas, entre 0,964 y 0,980, y la desviación típica interna de cada candidata, entre 0,013 y 0,020, es del mismo orden que las diferencias entre ellas.

Por tanto, esta comparación no sirve para elegir un modelo, solo para ver cómo se comportan con datos de entrenamiento.

## Conclusión sobre Optuna
Optuna haría una búsqueda guiada, útil para la primera explicación pero no para la segunda, porque solo encontraría mejor la candidata con mejor F1 macro interna, y esa F1 macro es ruidosa.

Por lo que decidimos no aplicar Optuna. Esto no demuestra que no ayude, porque solo hemos probado otra seed y más candidatas aleatorias, y todo se ha medido con la F1 macro.

## Robustez de la sensibilidad al umbral

Vamos a repetir con cada variante el recuento de casos del notebook 0.15, que reparte los 12 casos (estrategia y perfil) de cada hipótesis entre «no refutada», «depende del modelo» y «refutada» según cuántos de los cuatro modelos quedan por debajo del 80 % de completo.

La regresión logística no se ha repetido, así que es la misma en A, B y C.

In [ ]:

from urban_accessibility_ml.config import MATRIX_MODELS, THRESHOLDS
from urban_accessibility_ml.modeling.evaluate import cases_below, summarize_folds

CASE = ["strategy", "feature_set", "profile"]
ALL_MODELS = [m for m in MATRIX_MODELS if m != "dummy"]
HYPOTHESES = {"H1": "cartografico_aereo", "H2": "calle", "H5": "combinado"}
STATES = ["no refutada", "depende del modelo", "refutada"]
sensitivity = {
    "A": pd.read_csv(RESULTS / "sensitivity.csv"),
    "B": pd.read_csv(VARIANTS_DIR / "sensitivity_seed2.csv"),
    "C": pd.read_csv(VARIANTS_DIR / "sensitivity_n100.csv"),
}


def flags(summary, feature_set):
    """True if the case (strategy, profile) is below 80 % of completo, per model."""
    idx = summary[summary["feature_set"] == feature_set].set_index(["strategy", "profile"]).index.unique()
    out = pd.DataFrame(False, index=idx, columns=ALL_MODELS)
    for m in ALL_MODELS:
        below = cases_below(summary, m)
        below = below[below["feature_set"] == feature_set]
        out.loc[list(zip(below["strategy"], below["profile"])), m] = True
    return out


def verdicts(f):
    n = f.sum(axis=1)
    return n.map(lambda k: STATES[0] if k == 0 else STATES[2] if k == len(ALL_MODELS) else STATES[1])


below = {}  # (variant, hypothesis, threshold) -> flags of the 12 cases
for variant, matrix in zip("ABC", matrices.values()):
    completo = summarize_folds(matrix, CASE + ["model"]).query("feature_set == 'completo'")
    for t in THRESHOLDS:
        table = sensitivity[variant][sensitivity[variant]["threshold"] == t]
        summary = pd.concat([completo, summarize_folds(table, CASE + ["model"])])
        for h, feature_set in HYPOTHESES.items():
            if feature_set in summary["feature_set"].values:
                below[variant, h, t] = flags(summary, feature_set)

counts = pd.concat(
    {
        v: pd.DataFrame({(h, t): verdicts(f).value_counts().reindex(STATES, fill_value=0) for (w, h, t), f in below.items() if w == v}).T
        for v in "ABC"
    },
    axis=1,
)
counts.index.names = ["hipótesis", "umbral"]
counts.sort_index()

Observamos que en H1 con el 40 %, B y C tienen 7 casos que no se refutan y 5 que dependen del modelo, frente a 8 y 4 en A. Con el 50 % y el 60 %, B tiene 2 casos que no se refutan, 9 que dependen del modelo y 1 refutado, y C coincide con A.

En H2 con el 60 % y el 75 %, C tiene 1 caso que no se refuta, 10 que dependen del modelo y 1 refutado, frente a 1, 9 y 2 en A y en B. En H5 pasa lo mismo con el 75 %.

Con el 90 %, en H2 y H5 las tres variantes tienen 4 casos que dependen del modelo y 8 refutados.

El reparto de los 12 casos coincide en A, B y C en 7 de las 13 filas de la tabla. En las otras 6 cambia de grupo un solo caso.

La tabla anterior cuenta cuántos casos hay en cada grupo, pero no dice si son los mismos casos en A, B y C. Por eso ahora vamos a mostrar, por modelo, en cuántos casos cambia el resultado de B y de C respecto a A.

Contamos un caso cuando el modelo queda por debajo del 80 % de completo en una variante y no en A, o al revés.

In [ ]:
keys = [(h, t) for (v, h, t) in below if v == "A"]
changes = pd.DataFrame({v: sum((below[v, h, t] != below["A", h, t]).sum() for h, t in keys) for v in "BC"})
changes.rename_axis(f"casos que cambian, de {len(keys) * 12} por modelo")

Con B cambian 2 casos de `random_forest`, 4 de XGBoost y 10 de LightGBM. Con C cambian 3, 5 y 7, respectivamente.

La regresión logística no cambia en ninguno. Todos los cambios son de modelos de árboles, sobre todo de LightGBM.

## Resumen
- En los árboles el ajuste no mejora a los valores por defecto.
- Con más candidatas (C) sube la F1 macro interna, pero no la del test, así que la búsqueda no parece el problema.
- Con otra seed (B) cambian los resultados tanto como el propio ajuste, lo que apunta al ruido de la validación interna.
- De momento no hemos añadido Optuna.
- En la sensibilidad al umbral, el reparto de los casos de H1, H2 y H5 con B y con C coincide con el de A en 7 de las 13 filas, y en las otras 6 cambia un solo caso. Los cambios son de modelos de árboles.
- La comparación en el test entre los valores por defecto y los ajustados está en el [notebook 0.12](../0.12-tuned-experiment-matrix/).